# A watchlist supplied only by an SQL join

The original QuackWeave release transferred every author–message edge before DuckDB joined a five-row watchlist. This notebook submits the same public SQL to the production implementation. It checks each author, count, and fingerprint against direct Ladybug and verifies the number of rows delivered by the graph scan in `EXPLAIN ANALYZE`.

Run against the pinned 526,757,888-byte LSQB SF1 graph on macOS arm64. The execution cell launches the project's DuckDB 2.0.1 ADBC driver and Ladybug 0.22.3 in separate processes. It requires the built release, graph, and ADBC/PyArrow/Ladybug environment at the paths shown below. The baseline package from commit `ecfcd8e` is optional for reruns. The chart uses the notebook's DuckDB analysis kernel. [Benchmark runner](watchlist.py)

In [2]:
import json, subprocess
from pathlib import Path

here = Path.cwd().resolve()
root = next(p for p in (here, *here.parents) if (p / "benchmarks/lsqb_sf1/watchlist.py").exists())
work = root / "build/perf/lsqb-sf1"
baseline = work / "baseline-ecfcd8e/quack_weave.duckdb_extension"
argv = [str(root / "build/usage-client-venv/bin/python"), "-B",
        str(root / "benchmarks/lsqb_sf1/watchlist.py"),
        "--graph", str(work / "sf1.lbdb"),
        "--duckdb-library", str(root / "build/deps/duckdb-v2.0.1/libduckdb.dylib"),
        "--extension", str(root / "build/mvp2/release/extension/quack_weave/quack_weave.duckdb_extension")]
if baseline.exists():
    argv.extend(["--baseline", str(baseline)])
report = json.loads(subprocess.check_output(argv, text=True, cwd=root))
(work / "watchlist-release.json").write_text(json.dumps(report, indent=2))
print(report["sql"])
print("Direct Ladybug and extension results verified.", report["engines"]["release"]["rows"])

SELECT g.author, count(*) AS n, sum(g.mid % 1000003) AS fingerprint
FROM GRAPH_TABLE (social MATCH (m:Message)-[r:Message_hasCreator_Person]->(p:Person) COLUMNS (p.PersonId AS author, m.MessageId AS mid)) AS g
JOIN (VALUES (4398046519825::BIGINT), (2435::BIGINT), (6575::BIGINT), (28587302326226::BIGINT), (-1::BIGINT)) AS watch(author) ON watch.author = g.author
GROUP BY g.author ORDER BY g.author
Direct Ladybug and extension results verified. [[2435, 5721, 2736676126], [6575, 5660, 2913642663], [4398046519825, 6465, 3312328238], [28587302326226, 5470, 2740912288]]


In [3]:
WITH report AS (
    SELECT first(content)::JSON AS doc
    FROM read_text(['../../build/perf/lsqb-sf1/watchlist-release.json',
                    'build/perf/lsqb-sf1/watchlist-release.json'])
)
SELECT s.key AS engine, round((s.value->>'p50_ms')::DOUBLE, 2) AS median_ms,
       round((s.value->>'p95_ms')::DOUBLE, 2) AS p95_ms,
       (s.value->>'provider_rows')::BIGINT AS provider_rows,
       json_array_length(s.value->'rows') AS result_groups
FROM report, LATERAL json_each(doc, '$.engines') s ORDER BY engine;

engine,median_ms,p95_ms,provider_rows,result_groups
baseline,477.88,608.44,3809607,4
ladybug,3.52,4.26,NULL,4
release,8.33,8.8,23316,4


## What crossed into DuckDB

In the recorded 12 warm runs per engine, median latency fell from 477.88 ms to 8.33 ms. The graph scan emitted 23,316 matching edges across the four authors present in the data. The baseline emitted 3,809,607 edges. DuckDB still performed the SQL join and aggregation. Direct Ladybug filtered and aggregated within one engine, returning four rows in 3.52 ms.

The release recognizes a directly projected `INT64` property joined to a literal, single-column `VALUES` list of up to 64 rows. SQL keeps duplicate watchlist entries and determines NULL and outer-join semantics. A watchlist read from a DuckDB table or Parquet file uses the ordinary plan. These measurements describe a single machine and dataset; the regression test checks result equality and scan cardinality independently of latency.